# NumPy, pandas and Matplotlib

Previously you wrote loops by hand to compute an average, a maximum and a minimum. That's the right way to *learn*, but real physics data can have many **millions** of numbers. Now you'll meet three libraries that every scientist uses:

| Library | What it's for | Think of it as... |
|---|---|---|
| **NumPy** | Fast maths on whole arrays of numbers | A super-powered list |
| **pandas** | Tables of data (rows and columns) | Excel, but in Python |
| **Matplotlib** | Making plots | Easiest way to visualise your data |

In our simulations we stores particle positions and momenta as **NumPy arrays**, and we often analyse and plot simulation results with pandas and Matplotlib. So everything in this notebook feeds directly into the simulations you'll do later on.

### This notebook's plan
1. **NumPy warm-up:** create a range of momentum values and compute its statistics.
2. **pandas:** load a table of (fake) detector measurements, filter it, and compute statistics by group.
3. **Matplotlib:** draw a histogram and a scatter plot of the measurements.

Run cells top to bottom with **Shift + Enter**, and fill in the `# TODO`s.

## Setup

A **library** is a collection of code someone else wrote that we can use. We load libraries with `import`. The short names `np`, `pd` and `plt` are what everyone uses, so you'll see them in every tutorial online.

Run the cell below. If you get `ModuleNotFoundError`, ask a mentor: the library needs to be installed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("NumPy version:", np.__version__)
print("pandas version:", pd.__version__)

---
# Part 1 — NumPy warm-up

### Lists vs. arrays
A NumPy **array** looks like a list, but you can do maths on *all* its values at once, with no loop needed. \Run the cell below and compare what happens with a list vs. an array:

In [ ]:
energies_list = [3.2, 7.5, 1.1, 12.8]       # a normal Python list (from Day 1)
energies_array = np.array(energies_list)     # the same numbers as a NumPy array

print("List  * 2:", energies_list * 2)       # repeats the list!
print("Array * 2:", energies_array * 2)      # doubles every value

# Converting ALL energies from GeV to joules in one line (1 GeV = 1e9 eV):
EV_IN_JOULES = 1.602176634e-19
print("In joules:", energies_array * 1e9 * EV_IN_JOULES)

# Yesterday's loops, in one word each:
print("Mean:", energies_array.mean(), " Max:", energies_array.max(), " Min:", energies_array.min())

## Exercise 1 — An array of momenta

In a simulation we often want to scan a **range** of values, e.g. "try every momentum from 1 to 100 GeV/c".
The function `np.linspace(start, stop, num)` creates `num` **evenly spaced** values from `start` to `stop` (both included):
```python
np.linspace(0, 1, 5)    # array([0.  , 0.25, 0.5 , 0.75, 1.  ])
```

Two useful statistics:
- The **mean** is the average.
- The **standard deviation** ("std") measures how **spread out** values are around the mean.
  Small std means the values are bunched together; large std means they're spread out.
  In accelerators, the std of particle positions tells you **how wide the beam is**!

NumPy has them built in: `np.mean(arr)` and `np.std(arr)` (or `arr.mean()` and `arr.std()`).

### Your tasks:

1. Create `momenta`: an array of **300** evenly spaced values from **1** to **100** (GeV/c).
2. Store the number of values in `n_values` (hint: `len(momenta)` or `momenta.size`).
3. Compute `mean_momentum` and `std_momentum`.
4. Print the first and last values. Indexing works like lists: `momenta[0]` is the first; `momenta[-1]` is the **last**.

In [ ]:
# TODO 1: create the array
momenta = None

# TODO 2: how many values?
n_values = None

# TODO 3: mean and standard deviation
mean_momentum = None
std_momentum = None

# TODO 4: print the first and last value, plus your statistics

## Bonus (optional): a simulated beam.

Real particle beams aren't evenly spaced: most particles sit near the centre,
with fewer and fewer further out. This shape is called a **Gaussian** (or "normal" or "bell curve") distribution.

`rng.normal(loc=mean, scale=std, size=n)` makes `n` random values with a chosen mean and std:
```python
rng = np.random.default_rng(seed=42)   # a random number generator; the seed makes results repeatable
```
Create the horizontal positions `x` (in mm) of **10 000** particles with mean 0 and std 1.5.
Then compute the mean and std of your array. Are they close to 0 and 1.5? What happens with only 10 particles?

In [ ]:
# Bonus space

---
# Part 2 — pandas: tables of particle data

Detectors save their measurements in files. A very common format is **CSV** ("comma-separated values"): a plain text file where each line is a row and commas separate the columns. The first line holds the column names.

Our (fake!) detector measured 50 particles. For each one we have:

| Column | Meaning |
|---|---|
| `event_id` | A number identifying the measurement |
| `particle_type` | electron, muon, pion or proton |
| `detector` | Which of our two detectors (`A` or `B`) saw it |
| `momentum_GeV` | Momentum, in GeV/c |
| `energy_GeV` | Energy, in GeV |
| `x_mm`, `y_mm` | Where it was measured, relative to the beam centre, in mm |

Run the cell below. It writes the data into a file called `particle_data.csv` next to this notebook.
Notice how a CSV file is just text!

In [ ]:
csv_text = """event_id,particle_type,detector,momentum_GeV,energy_GeV,x_mm,y_mm
1,electron,B,3.34,3.34,1.68,0.28
2,pion,B,5.51,5.512,-0.69,0.46
3,pion,A,9.42,9.421,-1.02,-0.54
4,muon,A,6.41,6.411,1.56,1.04
5,muon,A,2.13,2.133,1.08,-0.69
6,proton,B,12.52,12.555,2.09,-1.42
7,proton,A,1.13,1.469,0.32,1.14
8,electron,A,1.42,1.42,2.44,-0.34
9,pion,A,0.22,0.261,-0.42,0.03
10,muon,B,12.2,12.2,-1.59,1.54
11,proton,B,2.64,2.802,-3.06,0.91
12,proton,B,1.79,2.021,-1.65,0.51
13,pion,A,2.49,2.494,1.23,0.86
14,proton,B,1.43,1.71,-2.2,-0.58
15,pion,B,9.55,9.551,-0.68,0.6
16,proton,A,15.73,15.758,3.16,0.89
17,muon,A,2.18,2.183,1.27,0.6
18,muon,B,14.03,14.03,2.69,-0.16
19,muon,B,6.27,6.271,-1.49,-3.0
20,electron,A,3.04,3.04,-3.69,0.45
21,pion,B,8.25,8.251,2.27,0.09
22,muon,B,3.54,3.542,-2.13,-0.3
23,pion,B,3.39,3.393,0.14,0.62
24,electron,A,11.14,11.14,0.33,0.21
25,muon,A,6.28,6.281,-1.42,0.16
26,muon,A,0.78,0.787,2.33,-1.19
27,electron,A,3.8,3.8,2.32,-1.17
28,pion,A,0.93,0.94,0.83,0.3
29,muon,A,4.4,4.401,-0.04,0.06
30,muon,A,2.38,2.382,0.29,0.11
31,electron,B,1.8,1.8,-1.64,0.13
32,proton,B,2.77,2.925,-0.36,0.33
33,proton,A,1.38,1.669,-0.16,0.03
34,pion,A,0.44,0.462,-1.28,0.13
35,muon,B,3.04,3.042,1.27,0.2
36,proton,B,1.44,1.719,-2.14,0.74
37,pion,B,4.22,4.222,-0.68,-0.41
38,muon,B,5.33,5.331,-3.38,0.63
39,proton,B,41.01,41.021,0.75,0.51
40,muon,B,2.27,2.272,1.11,0.08
41,pion,B,22.37,22.37,0.93,-0.38
42,muon,A,2.99,2.992,4.38,0.3
43,muon,B,12.3,12.3,1.41,0.69
44,pion,A,2.68,2.684,-1.41,-0.86
45,electron,B,7.24,7.24,4.0,0.05
46,pion,A,1.78,1.785,-1.38,0.05
47,pion,A,2.59,2.594,1.18,-0.53
48,electron,A,4.03,4.03,1.11,-0.5
49,pion,A,20.21,20.21,-1.32,-1.24
50,pion,B,1.21,1.218,2.09,-0.88
"""

with open("particle_data.csv", "w") as f:
    f.write(csv_text)

print("Saved particle_data.csv")

## Exercise 2.1 — Load the data

A table in pandas is called a **DataFrame**. Load a CSV file into one with:
```python
df = pd.read_csv("some_file.csv")
```

### Your tasks:

1. Load `particle_data.csv` into a DataFrame called `df`.
2. Store the number of rows in `n_rows` (hint: `len(df)` works, just like for a list).
3. Display the first 5 rows with `df.head()`. In a notebook, if the **last line** of a cell is a value, it gets displayed automatically as a nice table.

In [ ]:
# TODO 1: load the CSV file
df = None

# TODO 2: count the rows
n_rows = None

# TODO 3: display the first rows (put it on the last line of the cell!)

### A quick tour of pandas

Run these cells (they need `df` from the exercise above) to see some handy tools:

In [ ]:
# The size of the table: (rows, columns)
print(df.shape)

# The column names
print(df.columns)

In [ ]:
# Pick ONE column with square brackets. The result behaves like a NumPy array.
energy = df["energy_GeV"]
print("Mean energy:", energy.mean())
print("Max energy: ", energy.max())

In [ ]:
# describe() gives a quick summary of every numeric column
df.describe()

## Exercise 2.2 — Filtering: keep only the rows you want

Often we only care about *some* particles, for example the high-energy ones. pandas filters rows with a **condition**:
```python
df["energy_GeV"] > 5          # gives True/False for every row
df[df["energy_GeV"] > 5]      # keeps only the rows where it's True
```
It reads a bit strangely at first: "from `df`, give me the rows where `df["energy_GeV"] > 5`".

For text columns, use `==`, e.g. `df["particle_type"] == "muon"`.

### Your tasks:

1. Create `high_energy`: only the particles with **energy above 5 GeV**.
2. Store how many there are in `n_high_energy`.
3. Create `muons`: only the rows where `particle_type` is `"muon"`.

### Bonus:

Combine two conditions with `&` ("and"). Each condition needs its own brackets:
`df[(condition 1) & (condition 2)]`. Find all **protons** seen by **detector A**.

In [ ]:
# TODO 1: particles with energy above 5 GeV
high_energy = None

# TODO 2: how many are there?
n_high_energy = None

# TODO 3: only the muons
muons = None

In [ ]:
# Bonus space

## Exercise 2.3 — Grouped statistics

Are protons more energetic on average than electrons in our data? To find out, we **group** the rows by particle type and compute a statistic for each group. With `groupby` this takes one line:
```python
df.groupby("detector")["momentum_GeV"].mean()
```
Read it as: "group the rows by `detector`, take the `momentum_GeV` column, and compute the mean for each group".
Try running that line first to see what it gives!

### Your tasks:

1. Compute `mean_energy_by_type`: the **mean energy** for each `particle_type`.
2. Compute `count_by_type`: **how many** particles of each type there are (hint: `.size()` counts rows in each group, or try `df["particle_type"].value_counts()`).
3. Look at the result for protons with `mean_energy_by_type["proton"]`.

### Bonus:

`.agg(["mean", "min", "max"])` computes several statistics at once.
Try it on `momentum_GeV`, grouped by `detector`.

In [ ]:
# TODO 1: mean energy for each particle type
mean_energy_by_type = None

# TODO 2: number of particles of each type
count_by_type = None

print(mean_energy_by_type)
print(count_by_type)

---
# Part 3 — Plotting with Matplotlib

A good plot **always** has:
- an **x-axis label** (with units!),
- a **y-axis label** (with units!),
- a **title**.

A plot without labels is like a map without street names. Here's a complete example. Run it, then try changing things:

In [ ]:
x = np.linspace(0, 10, 100)
y = x ** 2

plt.figure(figsize=(6, 4))       # create a new, empty figure (width, height in inches)
plt.plot(x, y)                   # draw a line through the points
plt.xlabel("Time [s]")           # label the x-axis
plt.ylabel("Distance [m]")       # label the y-axis
plt.title("A ball rolling down a slope")
plt.grid(True)                   # optional: add grid lines
plt.show()                       # display the figure

## Exercise 3.1 — Histogram of particle energies

A **histogram** shows how values are *distributed*. It splits the range into "bins" (bars) and counts how many values land in each bin. It answers questions like *"are most particles low energy or high energy?"*
```python
plt.hist(values, bins=20)
```

### Your task:

Make a histogram of the `energy_GeV` column of `df`, with an x-label, a y-label and a title.

Then **experiment**: what changes when you use `bins=5` or `bins=50`? Try adding `color="orange"` or `edgecolor="black"`.

In [ ]:
plt.figure(figsize=(6, 4))

# TODO: draw the histogram of df["energy_GeV"]

# TODO: add an x-label, a y-label and a title

plt.show()

## Exercise 3.2 — Scatter plot: momentum vs. energy

A **scatter plot** draws one dot per particle, with one column on the x-axis and another on the y-axis. It's great for spotting **relationships** between two quantities.
```python
plt.scatter(x_values, y_values)
```

### Your task:

Make a scatter plot with `momentum_GeV` on the x-axis and `energy_GeV` on the y-axis.
Add axis labels (with units!) and a title.

Look at the result: what do you notice? (for fast particles, energy and momentum are almost equal.
You can see which particles are "slow" because they sit above the line!)

### Bonus 1:

Colour the dots by particle type and add a legend. Loop over the groups:
```python
for particle_type, group in df.groupby("particle_type"):
    plt.scatter(group["momentum_GeV"], group["energy_GeV"], label=particle_type)
plt.legend()
```
Try zooming in with `plt.xlim(0, 3)` and `plt.ylim(0, 3)`. Which particle type sits furthest above the line?

### Bonus 2: The beam spot.

Make a scatter plot of `x_mm` vs `y_mm`. This shows where particles hit the detector,
and is very close to the plots you'll make with XSuite! Add `plt.axis("equal")` so 1 mm looks the same in both directions. Is the beam wider horizontally or vertically?

In [ ]:
plt.figure(figsize=(6, 4))

# TODO: draw the scatter plot of momentum (x) vs energy (y)

# TODO: add an x-label, a y-label and a title

plt.show()

In [ ]:
# Bonus space